# 03 · Elo Baseline

**Goal:** Rate every NFL team with Elo and score its pre-game win probabilities against actual results.

**Data:** `nflmodel.data.read_games()`, filtered to `FIRST_SEASON`–`LAST_SEASON`

**Contents:** 0. Setup & parameters · 1. Load · 2. Elo math · 3. Elo engine · 4. Reshape ·
pick accuracy · 5. Brier score

Builds a baseline Elo rating for every NFL team, updated game by game and regressed toward the mean each offseason. All settings live in Section 0, so the model runs for any range of seasons. The output is a pre-game win probability for every game, scored against actual results.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from nflmodel.data import read_games

In [2]:
# =============================
# Section 0: setup & parameters
# =============================

# Timeframe
FIRST_SEASON = 2005
LAST_SEASON = 2025
SCORE_FROM = 2013
TRAIN_SEASONS = range(2013, 2022)
TEST_SEASONS = range(2022, 2026)

# Old Model Parameters
old_params = {
    "base": 1500,
    "scale": 400, 
    "k": 20,
    "hfa": 48,
    "season_regression": 1/3,
    "use_mov": True,
    "mov_scale": 2.2,
    "mov_slope": 0.001,
    "neutral_hfa": 0,
    "include_playoffs": True,
    "spread_divisor": 25
}
# Updated model parameters after tuning
params = {
    "base": 1500,
    "scale": 400, 
    "k": 21,
    "hfa": 42,
    "season_regression": 0.4,
    "use_mov": True,
    "mov_scale": 2.2,
    "mov_slope": 0.001,
    "neutral_hfa": 0,
    "include_playoffs": True,
    "spread_divisor": 25
}

In [3]:
# =============================
# Section 1: load & clean
# =============================

games = read_games()
games = games[games["season"].between(FIRST_SEASON, LAST_SEASON)].reset_index(drop=True)

In [4]:
# =============================
# Section 2: elo math
# =============================

def elo_diff(rating_home, rating_away, hfa):
    """
    Home team's Elo advantage, including home field
    
    rating_home: home team's Elo rating before the game
    rating_away: away team's Elo rating before the game
    hfa:         home-field advantage in Elo points (0 at neutral sites)
    """
    d = rating_home - rating_away + hfa
    return d

def home_win_prob(d, scale):
    """
    Probability the home team wins
    
    d:      home team's Elo advantage, from elo_diff
    scale:  Elo points that make 10-to-1 odds (400 by convention)
    """
    home_prob = 1 / (1 + 10 ** (-d / scale))
    return home_prob

def predicted_spread(d, spread_divisor):
    """ 
    Expected home margin in points

    d:              home team's Elo advantage, from elo_diff
    spread_divisor: Elo points per point of spread (25 convention)    
    """
    spread = d / spread_divisor
    return spread

def mov_multiplier(margin, d_winner, use_mov, mov_scale, mov_slope):
    """ 
    Scales the rating update by margin of victory

    margin:         home score minus away score
    d_winner:       winner's Elo advantage(d if home won, -d if away won)
    use_mov:        True to apply the multiplier, False to return 1
    mov_scale:      constant that sets the size of the favorite correction (2.2 convention)
    mov_slope:      how strongly the winner's Elo advantage shrinks the multiplier (0.001 convention)
    """
    if margin == 0:
        return 1.0
    if not use_mov:
        return 1.0
    multiplier = np.log(abs(margin) + 1) * mov_scale / (mov_slope * d_winner + mov_scale)
    return multiplier

def rating_change(k, multiplier, outcome, expected):
    """ 
    Elo points the home team gains (away team loses the same amount)
    
    k:          base update size (20 convention)
    multiplier: margin of victory scaling, from mov_multiplier
    outcome:    home result (1 win, 0.5 tie, 0 loss)
    expected:   home win probability, from home_win_prob
    """
    change = k * multiplier * (outcome - expected)
    return change

def regress_ratings(ratings, frac, base):
    """ 
    Pull every rating part of the way back toward base
    
    ratings:    dictionary of team's Elo ratings {team: elo rating}
    frac:       fraction of the gap to close (1/3 convention)
    base:       league-average rating to pull toward (1500 convention)
    """
    new_ratings = {team: r + frac * (base - r) for team, r in ratings.items()}
    return new_ratings


In [5]:
# =============================
# Section 3: elo engine
# =============================

def run_elo(games, params):
    """
    Run Elo through every game in kickoff order.

    games:  games DataFrame from Section 1, sorted by kickoff and game_id
    params: model parameter dictionary from Section 0

    Returns games with added columns:
    elo_home_pre, elo_away_pre, home_prob, elo_home_post, elo_away_post
    """

    # Drop playoff games if params states this
    if not params["include_playoffs"]:
        games = games[~games["is_playoff"]]

    ratings = {}                # team -> current Elo, the engine's memory
    current_season = None       # tracks when a new season starts
    results = []                # one dict per game, turned into columns at the end

    for g in games.itertuples(index=False):

        # 1. New season -> regress (skip for the first season as all teams start at 1500)
        if g.season != current_season:
            if current_season is not None:
                ratings = regress_ratings(ratings, params["season_regression"], params["base"])
            current_season = g.season

        # 2. Look up pre-game ratings (new teams start at base)
        r_home = ratings.get(g.home_team, params["base"])
        r_away = ratings.get(g.away_team, params["base"])

        # 3. Prediction
        hfa = params["neutral_hfa"] if g.is_neutral else params["hfa"]
        d = elo_diff(r_home, r_away, hfa)
        p_home = home_win_prob(d, params["scale"])

        # 4. Update only if game has been played
        r_home_post = r_home
        r_away_post = r_away
        if not pd.isna(g.home_win):
            d_winner = d if g.home_win >= 0.5 else -d
            mult = mov_multiplier(g.result, d_winner, params["use_mov"], params["mov_scale"], params["mov_slope"])
            change = rating_change(params["k"], mult, g.home_win, p_home)
            r_home_post = r_home + change
            r_away_post = r_away - change
            ratings[g.home_team] = r_home_post
            ratings[g.away_team] = r_away_post

        # 5. Save this game's numbers
        results.append({
            "elo_home_pre": r_home, 
            "elo_away_pre": r_away,
            "home_prob": p_home,
            "elo_home_post": r_home_post,
            "elo_away_post": r_away_post,
                        })
    out = pd.concat([games.reset_index(drop=True), pd.DataFrame(results)], axis=1)
    return out

In [6]:
# =============================
# Section 4: reshape
# =============================

def team_history(elo):
    """
    One row per team per game.

    elo: DataFrame returned by run_elo

    Returns columns: season, week, kickoff, team, opponent, elo_pre, elo_post
    """
    home = elo[["season", "week", "kickoff", "home_team", "away_team",
                "elo_home_pre", "elo_home_post"]].copy()
    home.columns = ["season", "week", "kickoff", "team", "opponent", "elo_pre", "elo_post"]

    away = elo[["season", "week", "kickoff", "away_team", "home_team", 
                "elo_away_pre", "elo_away_post"]].copy()
    away.columns = ["season", "week", "kickoff", "team", "opponent", "elo_pre", "elo_post"]

    history = pd.concat([home, away]).sort_values(["team", "kickoff"]).reset_index(drop=True)
    return history


def ratings_table(history, value):
    """
    Wide table: rows are (season, week), columns are teams.

    history: DataFrame from team_history
    value:   "elo_pre" or "elo_post"
    """
    table = history.pivot(index = ["season", "week"], columns = "team", values = value)
    table = table.ffill()
    return table

In [7]:
elo = run_elo(games, params)
elo.head(10)

,game_id,season,week,kickoff,game_type,is_playoff,home_team,away_team,location,is_neutral,...,home_qb_id,away_qb_id,spread_line,home_moneyline,away_moneyline,elo_home_pre,elo_away_pre,home_prob,elo_home_post,elo_away_post
0,2005_01_OAK_NE,2005,1,2005-09-08 09:00:00,REG,False,NE,LV,Home,False,...,00-0019596,00-0003292,7.5,NaN,NaN,1500.0,1500.0,0.56015,1521.734069,1478.265931
1,2005_01_CHI_WAS,2005,1,2005-09-11 13:00:00,REG,False,WAS,CHI,Home,False,...,00-0021379,00-0023541,6.0,NaN,NaN,1500.0,1500.0,0.56015,1509.957614,1490.042386
2,2005_01_CIN_CLE,2005,1,2005-09-11 13:00:00,REG,False,CLE,CIN,Home,False,...,00-0004293,00-0021429,-3.5,NaN,NaN,1500.0,1500.0,0.56015,1467.524810,1532.475190
3,2005_01_DEN_MIA,2005,1,2005-09-11 13:00:00,REG,False,MIA,DEN,Home,False,...,00-0005589,00-0013042,-4.5,NaN,NaN,1500.0,1500.0,0.56015,1529.175281,1470.824719
4,2005_01_HOU_BUF,2005,1,2005-09-11 13:00:00,REG,False,BUF,HOU,Home,False,...,00-0022912,00-0020608,4.5,NaN,NaN,1500.0,1500.0,0.56015,1525.130219,1474.869781
5,2005_01_NO_CAR,2005,1,2005-09-11 13:00:00,REG,False,CAR,NO,Home,False,...,00-0004161,00-0001823,6.5,NaN,NaN,1500.0,1500.0,0.56015,1483.375429,1516.624571
6,2005_01_NYJ_KC,2005,1,2005-09-11 13:00:00,REG,False,KC,NYJ,Home,False,...,00-0006355,00-0019559,3.0,NaN,NaN,1500.0,1500.0,0.56015,1527.594975,1472.405025
7,2005_01_SEA_JAX,2005,1,2005-09-11 13:00:00,REG,False,JAX,SEA,Home,False,...,00-0022177,00-0007091,3.0,NaN,NaN,1500.0,1500.0,0.56015,1523.248215,1476.751785
8,2005_01_TB_MIN,2005,1,2005-09-11 13:00:00,REG,False,MIN,TB,Home,False,...,00-0003739,00-0006423,6.0,NaN,NaN,1500.0,1500.0,0.56015,1470.200768,1529.799232
9,2005_01_TEN_PIT,2005,1,2005-09-11 13:00:00,REG,False,PIT,TEN,Home,False,...,00-0022924,00-0011024,7.0,NaN,NaN,1500.0,1500.0,0.56015,1530.202470,1469.797530


In [8]:
def check_rate(elo):
    """
    Adds a 'correct' column: True when the Elo favorite won.

    elo: DataFrame from run_elo
    """
    played = elo[(elo["result"] != 0) & elo["result"].notna()].copy()

    home_fav = played["home_prob"] > 0.5      # True where Elo picked the home team
    home_won = played["result"] > 0           # True where the home team won

    # HINT: the pick is correct when these two columns agree
    played["correct"] = home_fav == home_won
    return played

checked = check_rate(elo)

print(checked.groupby("home_team")["correct"].mean())
print(f"\nModel Accuracy: {checked.loc[checked["season"] >= SCORE_FROM, "correct"].mean()}")


home_team
ARI    0.578035
ATL    0.613636
BAL    0.681564
BUF    0.683333
CAR    0.590909
CHI    0.675978
CIN    0.574713
CLE    0.629412
DAL       0.625
DEN     0.61413
DET    0.728324
GB          0.7
HOU    0.646067
IND    0.651934
JAX    0.643678
KC     0.694737
LA     0.607955
LAC    0.657143
LV     0.602339
MIA    0.598837
MIN    0.657143
NE     0.751269
NO     0.622222
NYG    0.616279
NYJ    0.631579
PHI    0.672131
PIT    0.651934
SEA    0.722826
SF     0.653631
TB     0.640449
TEN    0.655172
WAS    0.566474
Name: correct, dtype: Float64

Model Accuracy: 0.6492957746478873


In [9]:
# =============================
# Section 5: evaluate
# =============================

def brier_score(df):
    """ 
    Average squared error of the home-win probabilites. 
    
    df: DataFrame with home_prob and home_win columns
    """
    played = df[df["home_win"].notna()]
    sq_err = (played["home_prob"] - played["home_win"]) ** 2
    brier = sq_err.mean()
    return brier

def brier_baselines(df):
    """ 
    Brier scores of two naive forecasters on the same games. 
    
    df: DataFrame with home_win column
    """
    played = df[df["home_win"].notna()]
    o_bar = played["home_win"].mean()
    coin_flip = 0.25
    home_rate = o_bar * (1 - o_bar)
    baselines = {"coin_flip": coin_flip, "home_rate": home_rate}
    return baselines

In [10]:
train = elo[elo["season"].isin(list(TRAIN_SEASONS))]
test = elo[elo["season"].isin(list(TEST_SEASONS))]

for name, df in [("train", train), ("test", test)]:
    b = brier_score(df)
    base = brier_baselines(df)
    bss = 1 - b / base["coin_flip"]
    print(f"{name}: Brier {b:.4f} | home-rate baseline {base["home_rate"]:.4f} | BSS {bss:.1%}")

train: Brier 0.2188 | home-rate baseline 0.2469 | BSS 12.5%
test: Brier 0.2212 | home-rate baseline 0.2471 | BSS 11.5%


In [11]:
# =============================
# Section 6: tune
# =============================

def tune(games, k_values, hfa_values, reg_values):
    """ 
    Training-season Brier score for every parameter combination. 
    
    games:      games DataFrame (full history, burn-in included)
    k_values:   list of K values to try
    hfa_values: list of HFA values to try
    reg_values: list of offseason regression values to try
    """

    rows = []
    for k in k_values:
        for hfa in hfa_values:
            for reg in reg_values:
                trial = {**params, "k": k, "hfa": hfa, "season_regression": reg}

                elo_t = run_elo(games, trial)
                train_t = elo_t[elo_t["season"].isin(list(TRAIN_SEASONS))]
                brier = brier_score(train_t)

                rows.append({"k":k, "hfa": hfa, "season_regression": reg, "brier": brier})
    results = pd.DataFrame(rows).sort_values("brier", ascending=True).reset_index(drop=True)
    return results

results = tune(games, 
               k_values = list(range(15, 34, 3)),
               hfa_values = list(range(24, 60, 2)),
               reg_values = [0, .1, .2, .3, .4, .5, .6, .7, .8, .9])
type(results)


pandas.DataFrame

In [12]:
print(results.head(10))
print(results.groupby("hfa")["brier"].min())
print(results.groupby("k")["brier"].min())
print(results.groupby("season_regression")["brier"].min())

    k  hfa  season_regression     brier
0  21   42                0.4  0.218781
1  21   40                0.4  0.218782
2  21   44                0.4  0.218794
3  21   38                0.4  0.218798
4  18   40                0.4  0.218817
5  18   42                0.4  0.218821
6  21   46                0.4  0.218821
7  21   36                0.4  0.218829
8  18   38                0.4  0.218829
9  18   44                0.4  0.218839
hfa
24    0.219322
26    0.219203
28    0.219098
30    0.219009
32    0.218934
34    0.218874
36    0.218829
38    0.218798
40    0.218782
42    0.218781
44    0.218794
46    0.218821
48    0.218863
50    0.218920
52    0.218990
54    0.219075
56    0.219174
58    0.219286
Name: brier, dtype: float64
k
15    0.219254
18    0.218817
21    0.218781
24    0.219062
27    0.219586
30    0.220217
33    0.221003
Name: brier, dtype: float64
season_regression
0.0    0.223722
0.1    0.221207
0.2    0.219692
0.3    0.218948
0.4    0.218781
0.5    0.219012
0.6    0.